# Final clean MMRDR-CFP experiment

Cleaned submission notebook. Outputs are intentionally stripped. The primary experiment pins code commit `0713e20bb81741163ffea2ee39a5be3dbb870b86`. Later repository commits add documentation and analysis exports only. Run top-to-bottom in Kaggle with the original MMRDR-CFP dataset, Internet ON and a GPU.

## Protocol

- Exact decoded-pixel duplicate handling precedes splitting.
- One fixed 85:15 development/validation split (seed 42).
- Four controlled ResNet-50 configurations: 224/384 px × head-only/partial fine-tuning.
- Primary selection metric: validation macro-F1; tie-breakers: QWK then fewer undergrades.
- Official test is evaluated only after selection is frozen and is reported as follow-up/non-blind because an earlier exploratory model had previously viewed that partition.
- XAI, lesion analyses and extra seeds are post-selection analyses and do not alter the primary model.

In [ ]:
# Reproducible final clean experiment notebook (outputs intentionally stripped)
from pathlib import Path
import json, subprocess, sys, shutil
import numpy as np, pandas as pd, torch
SEED=42
WORK=Path("/kaggle/working/final_clean_dr")
REPO=Path("/kaggle/working/clinically-explainable-dr")
MANIFESTS=WORK/"manifests"; ANALYSIS=WORK/"analysis"; FINAL_TEST=WORK/"final_test"; XAI_OUT=WORK/"xai"
for p in [WORK,MANIFESTS,ANALYSIS,FINAL_TEST,XAI_OUT]: p.mkdir(parents=True,exist_ok=True)
GITHUB_REPO="https://github.com/abhpasha/clinically-explainable-dr.git"
EXPERIMENT_COMMIT="0713e20bb81741163ffea2ee39a5be3dbb870b86"
if not REPO.exists(): subprocess.run(["git","clone",GITHUB_REPO,str(REPO)],check=True)
subprocess.run(["git","fetch","--all"],cwd=REPO,check=True)
subprocess.run(["git","checkout",EXPERIMENT_COMMIT],cwd=REPO,check=True)
subprocess.run([sys.executable,"-m","pip","install","-e",str(REPO),"--no-deps","-q"],check=True)
sys.path.insert(0,str(REPO/"src"))
print("Experiment code pinned at",EXPERIMENT_COMMIT)

In [ ]:
# Locate original MMRDR-CFP dataset and verify published CFP counts
INPUT=Path("/kaggle/input")
candidates=[]
for csv in INPUT.rglob("FP.csv"):
    img=csv.parent/"img"
    if img.is_dir(): candidates.append((csv,img))
assert candidates,"Attach MMRDR-CFP with FP.csv and img/"
CSV_PATH,IMAGES_DIR=candidates[0]; DATA_ROOT=CSV_PATH.parent
raw=pd.read_csv(CSV_PATH)
names=raw["image"].astype(str).map(lambda x:Path(x).name.lower())
tr=names.str.fullmatch(r"tr\d{6}\.jpe?g"); ts=names.str.fullmatch(r"ts\d{6}\.jpe?g")
assert len(raw)==11118 and tr.sum()==8893 and ts.sum()==2225
print(CSV_PATH, IMAGES_DIR, len(raw), int(tr.sum()), int(ts.sum()))

In [ ]:
# Leakage-controlled preparation BEFORE model fitting
from drx.clean_split import prepare as prepare_clean_split
summary=prepare_clean_split(CSV_PATH,DATA_ROOT,WORK/"clean_split",val_fraction=.15,seed=SEED)
for name in ["development.csv","validation.csv","official_test.csv"]:
    shutil.copy2(WORK/"clean_split"/name,MANIFESTS/name)
print(json.dumps(summary,indent=2))

In [ ]:
# Train the four pre-specified ResNet-50 configurations on identical manifests
CONFIGS=[
("A_head_224","configs/final_head_224_clean.yaml"),
("B_partial_224","configs/final_partial_224_clean.yaml"),
("C_head_384","configs/final_head_384_clean.yaml"),
("D_partial_384","configs/final_partial_384_clean.yaml"),
]
for name,cfg in CONFIGS:
    subprocess.run([
        sys.executable,"-m","drx.train","--config",str(REPO/cfg),
        "--train-csv",str(MANIFESTS/"development.csv"),
        "--val-csv",str(MANIFESTS/"validation.csv"),
        "--train-images",str(DATA_ROOT),"--filename-col","image","--label-col","grade"
    ],cwd=REPO,check=True)
print("Training stages complete")

In [ ]:
# Evaluate all four configurations on the same fixed validation set and select by the frozen rule
from torch.utils.data import DataLoader
from pathlib import Path
from sklearn.metrics import f1_score, cohen_kappa_score
from drx.core import build_model,device_choice,load_checkpoint
from drx.data import CFPDataset,parse_manifest
from drx.train import collect
from drx.metrics import summarise
device=device_choice()
val_manifest=parse_manifest(MANIFESTS/"validation.csv",DATA_ROOT,"image","grade")
dirs={
"A_head_224":REPO/"outputs/final_head_224_clean",
"B_partial_224":REPO/"outputs/final_partial_224_clean",
"C_head_384":REPO/"outputs/final_head_384_clean",
"D_partial_384":REPO/"outputs/final_partial_384_clean"}
rows=[]
for name,out in dirs.items():
    ck=load_checkpoint(out/"best.pt",device); m=build_model(ck["model_name"],False).to(device); m.load_state_dict(ck["model_state"]); m.eval()
    dl=DataLoader(CFPDataset(val_manifest,int(ck["image_size"])),batch_size=16 if int(ck["image_size"])==224 else 8,shuffle=False,num_workers=2)
    y,p,paths=collect(m,dl,device); r=summarise(y,p,ANALYSIS/name,[Path(x).name for x in paths])
    rows.append({"model":name,"accuracy":r["accuracy"],"macro_f1":r["macro_f1"],"qwk":r["quadratic_weighted_kappa"],"undergraded":r["undergraded"]})
comparison=pd.DataFrame(rows); comparison["macro_f1_selection"]=comparison.macro_f1.round(4)
comparison=comparison.sort_values(["macro_f1_selection","qwk","undergraded"],ascending=[False,False,True]).reset_index(drop=True)
SELECTED_MODEL=comparison.loc[0,"model"]; SELECTED_CHECKPOINT=dirs[SELECTED_MODEL]/"best.pt"
comparison.to_csv(ANALYSIS/"model_comparison_validation.csv",index=False)
print(comparison); print("Selected:",SELECTED_MODEL)

In [ ]:
# Freeze the selected model and evaluate the official test exactly once
import hashlib
def sha256(path):
    h=hashlib.sha256()
    with open(path,"rb") as f:
        for b in iter(lambda:f.read(1024*1024),b""): h.update(b)
    return h.hexdigest()
freeze={"selected_model":SELECTED_MODEL,"checkpoint_sha256":sha256(SELECTED_CHECKPOINT),"post_test_tuning_permitted":False,
        "qualification":"FOLLOW-UP / NON-BLIND official-test assessment"}
(FINAL_TEST/"frozen_before_test.json").write_text(json.dumps(freeze,indent=2))
test_manifest=parse_manifest(MANIFESTS/"official_test.csv",DATA_ROOT,"image","grade")
ck=load_checkpoint(SELECTED_CHECKPOINT,device); model=build_model(ck["model_name"],False).to(device); model.load_state_dict(ck["model_state"]); model.eval()
dl=DataLoader(CFPDataset(test_manifest,int(ck["image_size"])),batch_size=16 if int(ck["image_size"])==224 else 8,shuffle=False,num_workers=2)
ty,tp,tpaths=collect(model,dl,device)
summarise(ty,tp,FINAL_TEST,[Path(x).name for x in tpaths])
(FINAL_TEST/"qualification.json").write_text(json.dumps(freeze,indent=2))
print(freeze)

In [ ]:
# Reliability and validation-frozen clinician-review policies
pred_val=pd.read_csv(ANALYSIS/SELECTED_MODEL/"predictions.csv")
pred_test=pd.read_csv(FINAL_TEST/"predictions.csv")
def scores(df):
    P=df[[f"prob_{i}" for i in range(5)]].to_numpy(float)
    conf=P.max(1); ent=-(P*np.log(np.clip(P,1e-12,1))).sum(1); s=np.sort(P,axis=1); margin=s[:,-1]-s[:,-2]
    return {"one_minus_confidence":1-conf,"entropy":ent,"one_minus_top1_top2_margin":1-margin}
sv,st=scores(pred_val),scores(pred_test)
rows=[]
for method,vscore in sv.items():
    for frac in [.1,.2,.3]:
        threshold=float(np.quantile(vscore,1-frac,method="higher"))
        score=st[method]; review=score>=threshold
        y=pred_test.true_grade.to_numpy(int); p=pred_test.pred_grade.to_numpy(int)
        err=p!=y; under=p<y; pdr=(y==4)&(p<4)
        rows.append({"method":method,"target_review_fraction_from_validation":frac,"validation_threshold_frozen":threshold,
                     "actual_test_review_fraction":float(review.mean()),
                     "test_errors_captured_fraction":float((review&err).sum()/max(1,err.sum())),
                     "test_undergrades_captured_fraction":float((review&under).sum()/max(1,under.sum())),
                     "test_pdr_undergrades_captured_fraction":float((review&pdr).sum()/max(1,pdr.sum()))})
pd.DataFrame(rows).to_csv(ANALYSIS/"review_policy_test.csv",index=False)

## Quantitative XAI and post-hoc analyses

The following cells reproduce the report-level reliability, lesion and XAI analyses. The fundus-constrained random control is implemented here by restricting random 8×8 regions to cells with at least 50% estimated retinal-field occupancy; `src/drx/xai_checks.py` supplies perturbation and scoring primitives.

In [ ]:
# Lesion-conditioned undergrading with Haldane OR CIs and Benjamini-Hochberg correction
import ast,re
from scipy.stats import fisher_exact,norm
LESIONS=["Microaneurysm","Hard exudate","Intraretinal haemorrhage","Venous beading or IRMA","Neovascularisation","Vitreous haemorrhage","Retinal detachment"]
def parse_lesion(v):
    try: a=ast.literal_eval(str(v))
    except Exception: a=[int(x) for x in re.findall(r"(?<!\d)[01](?!\d)",str(v))]
    return [int(float(x)) for x in a]
ann=raw[["image","grade","lesion"]].copy(); ann["image"]=ann.image.map(lambda x:Path(str(x)).name)
lm=pd.DataFrame(ann.lesion.map(parse_lesion).tolist(),columns=LESIONS,index=ann.index); ann=pd.concat([ann[["image","grade"]],lm],axis=1)
pv=pred_val.copy(); pv["image"]=pv.path.map(lambda x:Path(str(x)).name); pv=pv.merge(ann,on="image",how="left"); pv["undergraded"]=pv.pred_grade<pv.true_grade
def or_ci(a,b,c,d):
    aa,bb,cc,dd=[x+.5 for x in [a,b,c,d]]; OR=(aa*dd)/(bb*cc); se=np.sqrt(1/aa+1/bb+1/cc+1/dd); z=norm.ppf(.975)
    return OR,np.exp(np.log(OR)-z*se),np.exp(np.log(OR)+z*se)
rows=[]
for grade,name in [(3,"Severe NPDR"),(4,"PDR")]:
    g=pv[pv.true_grade==grade]
    for lesion in LESIONS:
        present=g[lesion]==1; a=int((present&g.undergraded).sum()); b=int((present&~g.undergraded).sum())
        c=int((~present&g.undergraded).sum()); d=int((~present&~g.undergraded).sum()); OR,lo,hi=or_ci(a,b,c,d); _,p=fisher_exact([[a,b],[c,d]])
        rows.append({"true_grade":grade,"true_grade_name":name,"lesion":lesion,"n":len(g),"lesion_present_n":int(present.sum()),
                     "undergrade_rate_if_present":float(g.loc[present,"undergraded"].mean()) if present.any() else np.nan,
                     "undergrade_rate_if_absent":float(g.loc[~present,"undergraded"].mean()) if (~present).any() else np.nan,
                     "haldane_odds_ratio":OR,"odds_ratio_95_ci_low":lo,"odds_ratio_95_ci_high":hi,"fisher_p_raw":p})
out=pd.DataFrame(rows); order=np.argsort(out.fisher_p_raw.to_numpy()); m=len(out); adj=np.empty(m); running=1.
for rev,idx in enumerate(order[::-1],1):
    rank=m-rev+1; running=min(running,out.fisher_p_raw.iloc[idx]*m/rank); adj[idx]=running
out["fisher_p_bh_adjusted"]=np.clip(adj,0,1); out.to_csv(ANALYSIS/"lesion_conditioned_undergrading.csv",index=False)
consistency={"grade1_with_any_non_microaneurysm_lesion":int(((ann.grade==1)&(ann[LESIONS[1:]].max(1)==1)).sum()),
             "pdr_without_nv_vh_or_rd":int(((ann.grade==4)&(ann[["Neovascularisation","Vitreous haemorrhage","Retinal detachment"]].max(1)==0)).sum())}
(ANALYSIS/"label_lesion_consistency_audit.json").write_text(json.dumps(consistency,indent=2))

In [ ]:
# Quantitative XAI faithfulness audit: 10 validation images/grade, 8x8 grid, fundus-constrained random deletion
import torch.nn.functional as F
from PIL import Image
from torchvision.transforms.functional import gaussian_blur
from drx.data import transform
from drx.explain import gradcam
from drx.xai_checks import FRACTIONS,rank_regions,occlusion_importance,deletion_probabilities,mean_integrated_drop,spearman_map_correlation,paired_bootstrap
ck=load_checkpoint(SELECTED_CHECKPOINT,device); xmodel=build_model(ck["model_name"],False).to(device); xmodel.load_state_dict(ck["model_state"]); xmodel.eval()
size=int(ck["image_size"]); mean=torch.tensor([.485,.456,.406],device=device).view(3,1,1); std=torch.tensor([.229,.224,.225],device=device).view(3,1,1)
selected=pd.concat([g.sample(n=min(10,len(g)),random_state=SEED) for _,g in pd.read_csv(MANIFESTS/"validation.csv").groupby("grade")],ignore_index=True)
def loadx(fn):
    with Image.open(IMAGES_DIR/Path(fn).name) as im: return transform(size)(im.convert("RGB")).to(device)
def occ(mask,grid=8):
    h,w=mask.shape; return np.array([float(mask[(j//grid)*h//grid:(j//grid+1)*h//grid,(j%grid)*w//grid:(j%grid+1)*w//grid].float().mean()) for j in range(grid*grid)])
rows=[]
original_maps={}
for i,r in selected.iterrows():
    x=loadx(r.image); rgb=(x*std+mean).clamp(0,1)
    with torch.inference_mode(): pr=xmodel(x[None]).softmax(1)[0]; target=int(pr.argmax()); original=float(pr[target])
    baseline=(gaussian_blur(rgb,[41,41],[12.,12.])-mean)/std
    cam,_,_=gradcam(xmodel,x[None].detach().clone(),target_class=target); cg=F.adaptive_avg_pool2d(torch.tensor(cam,device=device)[None,None],(8,8))[0,0].cpu().numpy().ravel()
    mask=rgb.max(0).values>.06; occupancy=occ(mask); eligible=np.where(occupancy>=.5)[0]; fringe=np.where(occupancy<.5)[0]
    random_curves=[]
    for rep in range(10):
        rng=np.random.default_rng(SEED*100003+i*100+rep); rr=np.concatenate([rng.permutation(eligible),rng.permutation(fringe)])
        random_curves.append(deletion_probabilities(xmodel,x,baseline,rr,target,8,device,8))
    rnd=np.mean(np.stack(random_curves),0); grd=deletion_probabilities(xmodel,x,baseline,rank_regions(cg),target,8,device,8)
    oi=occlusion_importance(xmodel,x,baseline,target,original,8,device,8)
    rows.append({"image":r.image,"true_grade":int(r.grade),"pred_grade":target,"confidence":original,
                 "aop_drop_gradcam":mean_integrated_drop(grd),"aop_drop_fundus_random":mean_integrated_drop(rnd),
                 "gradcam_occlusion_spearman":spearman_map_correlation(cg,oi)})
xai=pd.DataFrame(rows); xai.to_csv(XAI_OUT/"xai_fundus_control_per_image.csv",index=False)
summary=paired_bootstrap(xai.aop_drop_gradcam,xai.aop_drop_fundus_random,seed=SEED,repetitions=2000)
(XAI_OUT/"xai_summary.json").write_text(json.dumps({"n":len(xai),"gradcam_mean":xai.aop_drop_gradcam.mean(),"random_mean":xai.aop_drop_fundus_random.mean(),"paired":summary},indent=2))

In [ ]:
# Cascading parameter-randomisation sanity check on 15 grade-spanning images
import copy
sanity=selected.groupby("grade",group_keys=False).head(3).head(15)
maps={}
for _,r in sanity.iterrows():
    x=loadx(r.image)
    with torch.inference_mode(): target=int(xmodel(x[None]).argmax(1))
    cam,_,_=gradcam(xmodel,x[None].detach().clone(),target_class=target); maps[r.image]=(x,target,cam)
state=copy.deepcopy(xmodel.state_dict())
def reset(module):
    for m in module.modules():
        if hasattr(m,"reset_parameters"): m.reset_parameters()
torch.manual_seed(SEED+777); xmodel.fc.reset_parameters(); xmodel.eval()
fc=[]
for fn,(x,t,c) in maps.items():
    z,_,_=gradcam(xmodel,x[None].detach().clone(),target_class=t); fc.append((fn,spearman_map_correlation(c,z)))
xmodel.load_state_dict(state); torch.manual_seed(SEED+778); reset(xmodel.layer4); xmodel.fc.reset_parameters(); xmodel.eval()
cascade=[]
for fn,(x,t,c) in maps.items():
    z,_,_=gradcam(xmodel,x[None].detach().clone(),target_class=t); cascade.append((fn,spearman_map_correlation(c,z)))
xmodel.load_state_dict(state); xmodel.eval()
pd.DataFrame({"image":[x[0] for x in fc],"fc_only_randomized_gradcam_spearman":[x[1] for x in fc],
              "layer4_plus_fc_randomized_gradcam_spearman":[x[1] for x in cascade]}).to_csv(XAI_OUT/"parameter_randomization_sanity.csv",index=False)

## Sensitivity analyses

Extra seeds 7 and 123 and the lesion-concept probe are post-hoc sensitivity analyses. They must not replace the primary seed-42 model after the official test has been viewed.

In [ ]:
# Post-hoc selected-model seed sensitivity (does not alter the primary seed-42 model)
# Re-run the selected 384 px staged training for seeds 7 and 123 using the same clean manifests.
import yaml
for extra_seed in [7,123]:
    head=yaml.safe_load((REPO/"configs/final_head_384_clean.yaml").read_text()); head["seed"]=extra_seed; head["output_dir"]=f"outputs/seed_{extra_seed}_head_384"
    hp=WORK/f"seed_{extra_seed}_head.yaml"; hp.write_text(yaml.safe_dump(head,sort_keys=False))
    subprocess.run([sys.executable,"-m","drx.train","--config",str(hp),"--train-csv",str(MANIFESTS/"development.csv"),"--val-csv",str(MANIFESTS/"validation.csv"),"--train-images",str(DATA_ROOT),"--filename-col","image","--label-col","grade"],cwd=REPO,check=True)
    part=yaml.safe_load((REPO/"configs/final_partial_384_clean.yaml").read_text()); part["seed"]=extra_seed; part["init_checkpoint"]=f"outputs/seed_{extra_seed}_head_384/best.pt"; part["output_dir"]=f"outputs/seed_{extra_seed}_partial_384"
    pp=WORK/f"seed_{extra_seed}_partial.yaml"; pp.write_text(yaml.safe_dump(part,sort_keys=False))
    subprocess.run([sys.executable,"-m","drx.train","--config",str(pp),"--train-csv",str(MANIFESTS/"development.csv"),"--val-csv",str(MANIFESTS/"validation.csv"),"--train-images",str(DATA_ROOT),"--filename-col","image","--label-col","grade"],cwd=REPO,check=True)

In [ ]:
# Post-hoc lesion concept probe: selected features vs frozen 384 px features and grade probabilities, including within-PDR analysis
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score,average_precision_score,balanced_accuracy_score
ann_idx=ann.set_index("image"); dev_names=pd.read_csv(MANIFESTS/"development.csv").image.tolist(); val_names=pd.read_csv(MANIFESTS/"validation.csv").image.tolist()
def feature_matrix(checkpoint,manifest):
    ck=load_checkpoint(checkpoint,device); m=build_model(ck["model_name"],False).to(device); m.load_state_dict(ck["model_state"]); m.eval(); m.fc=torch.nn.Identity()
    dl=DataLoader(CFPDataset(parse_manifest(manifest,DATA_ROOT,"image","grade"),int(ck["image_size"]),training=False),batch_size=8,shuffle=False,num_workers=2)
    X=[]; names=[]
    with torch.inference_mode():
        for xb,_,paths in dl: X.append(m(xb.to(device)).cpu().numpy()); names.extend([Path(p).name for p in paths])
    return np.concatenate(X),names
Xdev,nd=feature_matrix(SELECTED_CHECKPOINT,MANIFESTS/"development.csv"); Xval,nv=feature_matrix(SELECTED_CHECKPOINT,MANIFESTS/"validation.csv")
Xdev0,_=feature_matrix(dirs["C_head_384"]/"best.pt",MANIFESTS/"development.csv"); Xval0,_=feature_matrix(dirs["C_head_384"]/"best.pt",MANIFESTS/"validation.csv")
Ydev=ann_idx.loc[nd,LESIONS].to_numpy(int); Yval=ann_idx.loc[nv,LESIONS].to_numpy(int); gdev=ann_idx.loc[nd,"grade"].to_numpy(int); gval=ann_idx.loc[nv,"grade"].to_numpy(int)
Pdev=pd.read_csv(ANALYSIS/SELECTED_MODEL/"predictions.csv")[[f"prob_{i}" for i in range(5)]].to_numpy(float)
# For development probabilities, run selected classifier once
def probs(checkpoint,manifest):
    ck=load_checkpoint(checkpoint,device); m=build_model(ck["model_name"],False).to(device); m.load_state_dict(ck["model_state"]); m.eval()
    dl=DataLoader(CFPDataset(parse_manifest(manifest,DATA_ROOT,"image","grade"),int(ck["image_size"]),training=False),batch_size=8,shuffle=False,num_workers=2)
    out=[]
    with torch.inference_mode():
        for xb,_,_ in dl: out.append(m(xb.to(device)).softmax(1).cpu().numpy())
    return np.concatenate(out)
Pdev=probs(SELECTED_CHECKPOINT,MANIFESTS/"development.csv"); Pval=probs(SELECTED_CHECKPOINT,MANIFESTS/"validation.csv")
def probe(Xtr,Xva,ytr,yva):
    clf=make_pipeline(StandardScaler(),LogisticRegression(max_iter=1500,class_weight="balanced",random_state=SEED)); clf.fit(Xtr,ytr); p=clf.predict_proba(Xva)[:,1]
    return roc_auc_score(yva,p),average_precision_score(yva,p)
rows=[]; pdrd=gdev==4; pdrv=gval==4
for lesion in ["Neovascularisation","Vitreous haemorrhage","Retinal detachment"]:
    j=LESIONS.index(lesion)
    for source,A,B in [("selected_grade_probabilities",Pdev,Pval),("frozen_384_backbone_features",Xdev0,Xval0),("selected_partial_384_features",Xdev,Xval)]:
        auc,ap=probe(A[pdrd],B[pdrv],Ydev[pdrd,j],Yval[pdrv,j]); rows.append({"lesion":lesion,"source":source,"validation_pdr_n":int(pdrv.sum()),"validation_positive_n":int(Yval[pdrv,j].sum()),"roc_auc":auc,"average_precision":ap})
pd.DataFrame(rows).to_csv(ANALYSIS/"lesion_probe_within_pdr_comparison.csv",index=False)